In [2]:
import flappy_bird_env  # noqa
import numpy as np
import cv2
# Note: For render_mode="human", we don't need headless setup
# pygame will initialize automatically when creating the environment

import gymnasium as gym
from memory_system import MemoryConfig, MemoryBank, MemoryType
from instruction_set import InstructionSet
from operation import ALL_OPS, SCALAR_OPS, AUTOML_ALL_OPS, CV_ALL_OPS
from individual import Individual
from population import Population, PopulationConfig
from operators import GeneticOperators
from evaluator import CartPoleEvaluator, FlappyBirdEvaluator
from evolution_engine import EvolutionEngine, EvolutionConfig


In [3]:
# Setup for FlappyBird with patch-based image observations
rng = np.random.default_rng(0)

# Memory configuration for FlappyBird
# We need matrix observation registers to 
# store image patches
# Default patch size is 72x72, and we extract 8 patches
patch_size = 700
num_patches = 700

memory_cfg = MemoryConfig(
    n_scalar=8,          # Working scalars (for computation)
    n_vector=8,          # Working vectors
    n_matrix=8,          # Working matrices (for CV operations)
    n_obs_scalar=0,      # No scalar observations (using matrices instead)
    n_obs_vector=0,      # No vector observations
    n_obs_matrix=1,  # 8 matrix observation registers for patches
    vector_size=200,
    matrix_shape=(patch_size, patch_size),  # Each patch is 72x72
)

# Use AutoML + CV operations for image processing
# CV operations work on matrices (our patches)
# AutoML operations handle the logic/decision making
all_ops = AUTOML_ALL_OPS + CV_ALL_OPS
instruction_set = InstructionSet([op() for op in all_ops], memory_cfg)
operators = GeneticOperators(instruction_set, rng)

print(f"Memory config: {memory_cfg}")
print(f"Total operations: {len(all_ops)}")
print(f"  - AutoML operations: {len(AUTOML_ALL_OPS)}")
print(f"  - CV operations: {len(CV_ALL_OPS)}")


Memory config: MemoryConfig(n_scalar=8, n_vector=8, n_matrix=8, n_obs_scalar=0, n_obs_vector=0, n_obs_matrix=1, vector_size=200, matrix_shape=(700, 700), init_scalar_range=(-2.0, 2.0), init_vector_range=(-1.0, 1.0), init_matrix_range=(-0.5, 0.5))
Total operations: 76
  - AutoML operations: 64
  - CV operations: 12


In [4]:
# Create FlappyBird evaluator with patch-based observation processing
evaluator = FlappyBirdEvaluator(
    env_id="FlappyBird-v0",
    episodes=3,  # Test with 3 episodes
    max_steps=500,
    output_register=7,  # Read action from scalar register 7
    render_mode="human",  # Watch the game

    rng=rng,
    # Patch extraction parameters
    patch_size=200,
    num_patches=8,
    patch_strategy="full_image",  # Try "grid", "strategic", "overlapping", or "random"
    color_channel=1,  # Green channel (1), or -1 for grayscale
    normalize=True,
)

print("FlappyBird Evaluator created!")
print(f"  Patch size: {evaluator.patch_size}x{evaluator.patch_size}")
print(f"  Number of patches: {evaluator.num_patches}")
print(f"  Strategy: {evaluator.patch_strategy}")
print(f"  Color channel: {evaluator.color_channel} ({'grayscale' if evaluator.color_channel == -1 else ['R', 'G', 'B'][evaluator.color_channel]})")


FlappyBird Evaluator created!
  Patch size: 200x200
  Number of patches: 8
  Strategy: full_image
  Color channel: 1 (G)


In [5]:
# # Test patch extraction on a sample observation
# env = gym.make("FlappyBird-v0", render_mode="human")
# test_observation, _ = env.reset()
# test_observation = np.asarray(test_observation, dtype=np.float32)

# # Process observation into patches
# patches = evaluator._process_observation(test_observation)

# print(f"Extracted {len(patches)} patches")
# print(f"Each patch shape: {patches[0].shape}")
# print(f"Patch value range: [{patches[0].min():.3f}, {patches[0].max():.3f}]")

# # Visualize a few patches
# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(4, 4, figsize=(16, 8))
# for i, ax in enumerate(axes.flat):
#     if i < len(patches):
#         ax.imshow(patches[i], cmap='gray', vmin=0, vmax=1)
#         ax.set_title(f"Patch {i}")
#         ax.axis('off')
#     else:
#         ax.axis('off')
# plt.tight_layout()
# plt.show()

# env.close()


In [6]:
# # Visualize 700x576 crop (removing top 50 and bottom 50 pixels)
# env = gym.make("FlappyBird-v0", render_mode="human")
# test_observation, info = env.reset()
# test_observation = np.asarray(test_observation, dtype=np.float32)

# # Original image dimensions
# h, w = test_observation.shape[:2]  # 800x576
# print(f"Original image: {h}x{w}")

# # Crop parameters: remove top 50 and bottom 50 pixels
# crop_top = 50
# crop_bottom = 50
# new_height = h - crop_top - crop_bottom  # 800 - 50 - 50 = 700
# new_width = w  # Keep full width: 576

# print(f"\nCrop configuration:")
# print(f"  Remove top: {crop_top} pixels")
# print(f"  Remove bottom: {crop_bottom} pixels")
# print(f"  New dimensions: {new_height}x{new_width}")
# print(f"  Lost pixels: {crop_top + crop_bottom} rows ({100 * (crop_top + crop_bottom) / h:.1f}% of height)")

# # Extract color channel (green)
# single_channel = test_observation[:, :, 1].astype(np.float32) / 255.0

# # Create cropped version (700x576)
# cropped_700 = single_channel[crop_top:h-crop_bottom, :]

# # Get pipe information if available
# pipe_info = []
# if hasattr(env, 'env') and hasattr(env.env, 'pipes'):
#     pipes = env.env.pipes
#     for pipe in pipes:
#         pipe_x = getattr(pipe, 'x', 0)
#         pipe_top = getattr(pipe, 'top', 0)
#         pipe_bottom = getattr(pipe, 'bottom', 0)
#         pipe_info.append({
#             'x': pipe_x,
#             'top': pipe_top,
#             'bottom': pipe_bottom,
#             'visible_x': 0 <= pipe_x < w,
#             'visible_y': crop_top <= pipe_top < (h - crop_bottom) or crop_top <= pipe_bottom < (h - crop_bottom)
#         })

# # Visualize
# import matplotlib.pyplot as plt
# import matplotlib.patches as patches

# fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# # 1. Original image with crop boundaries
# ax1 = axes[0, 0]
# ax1.imshow(test_observation.astype(np.uint8))
# # Draw crop boundaries
# rect = patches.Rectangle((0, crop_top), w, new_height, 
#                         linewidth=3, edgecolor='lime', facecolor='none', linestyle='--')
# ax1.add_patch(rect)
# # Draw lost regions
# lost_top = patches.Rectangle((0, 0), w, crop_top, 
#                              linewidth=2, edgecolor='red', facecolor='red', alpha=0.3)
# lost_bottom = patches.Rectangle((0, h - crop_bottom), w, crop_bottom, 
#                                linewidth=2, edgecolor='red', facecolor='red', alpha=0.3)
# ax1.add_patch(lost_top)
# ax1.add_patch(lost_bottom)
# ax1.set_title(f"Original 800x576 with Crop Boundaries\nGreen dashed: kept region, Red: lost regions", 
#               fontsize=12, fontweight='bold')
# ax1.axis('off')

# # Mark pipe positions if available
# if pipe_info:
#     for pipe in pipe_info:
#         pipe_x = pipe['x']
#         if 0 <= pipe_x < w:
#             color = 'yellow' if pipe['visible_x'] and pipe['visible_y'] else 'red'
#             ax1.axvline(x=pipe_x, color=color, linewidth=2, linestyle=':', alpha=0.8)
#             if pipe['visible_x']:
#                 ax1.text(pipe_x, 100, f"Pipe\nx={pipe_x}", color=color, fontsize=8, 
#                         ha='center', bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

# # 2. What gets cropped out (top)
# ax2 = axes[0, 1]
# top_cropped = test_observation[:crop_top, :]
# if top_cropped.size > 0:
#     ax2.imshow(top_cropped.astype(np.uint8))
#     ax2.set_title(f"Lost: Top {crop_top} rows", 
#                   fontsize=12, fontweight='bold', color='red')
# else:
#     ax2.text(0.5, 0.5, "No top crop", ha='center', va='center', fontsize=14)
# ax2.axis('off')

# # 3. What gets cropped out (bottom)
# ax3 = axes[0, 2]
# bottom_cropped = test_observation[h-crop_bottom:, :]
# if bottom_cropped.size > 0:
#     ax3.imshow(bottom_cropped.astype(np.uint8))
#     ax3.set_title(f"Lost: Bottom {crop_bottom} rows", 
#                   fontsize=12, fontweight='bold', color='red')
# else:
#     ax3.text(0.5, 0.5, "No bottom crop", ha='center', va='center', fontsize=14)
# ax3.axis('off')

# # 4. The cropped version (what we keep)
# ax4 = axes[1, 0]
# ax4.imshow(cropped_700, cmap='gray', vmin=0, vmax=1)
# ax4.set_title(f"Kept: Cropped 700x576 region\nRows {crop_top} to {h-crop_bottom}", 
#               fontsize=12, fontweight='bold', color='green')
# ax4.axis('off')

# # Mark pipe positions in cropped view
# if pipe_info:
#     for pipe in pipe_info:
#         pipe_x = pipe['x']
#         if 0 <= pipe_x < w and pipe['visible_y']:
#             # Adjust y coordinates for cropped view
#             adjusted_top = pipe['top'] - crop_top
#             adjusted_bottom = pipe['bottom'] - crop_top
#             if 0 <= adjusted_top < new_height or 0 <= adjusted_bottom < new_height:
#                 ax4.axvline(x=pipe_x, color='yellow', linewidth=2, linestyle=':', alpha=0.8)
#                 ax4.text(pipe_x, 50, f"Pipe\nx={pipe_x}", color='yellow', fontsize=8, 
#                         ha='center', bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

# # 5. Show patches that would be extracted (with patch_size=200)
# ax5 = axes[1, 1]
# # For 700x576: we can't cleanly divide 700 by 200 (700/200 = 3.5)
# # Let's show what patches we'd get with patch_size=175: 700/175 = 4, so 4x2 = 8 patches
# # Or with patch_size=140: 700/140 = 5, 576/144 = 4, so 5x4 = 20 patches
# patch_size_700 = 140
# grid_h = new_height // patch_size_700  # 700/140 = 5
# grid_w = new_width // patch_size_700   # 576/140 = 4 (truncated, but let's use 144 for width)
# patch_size_w = 144
# grid_w = new_width // patch_size_w  # 576/144 = 4

# # Extract patches
# patches_700 = []
# for i in range(grid_h):
#     for j in range(grid_w):
#         y0 = i * patch_size_700
#         y1 = min(y0 + patch_size_700, new_height)
#         x0 = j * patch_size_w
#         x1 = min(x0 + patch_size_w, new_width)
#         patch = np.zeros((patch_size_700, patch_size_w), dtype=np.float32)
#         patch[:y1-y0, :x1-x0] = cropped_700[y0:y1, x0:x1]
#         patches_700.append(patch)

# # Create composite
# composite_700 = np.zeros((grid_h * patch_size_700, grid_w * patch_size_w), dtype=np.float32)
# for idx, patch in enumerate(patches_700):
#     i, j = idx // grid_w, idx % grid_w
#     composite_700[i*patch_size_700:(i+1)*patch_size_700, 
#                   j*patch_size_w:(j+1)*patch_size_w] = patch

# ax5.imshow(composite_700, cmap='gray', vmin=0, vmax=1)
# ax5.set_title(f"700x576 patches\n{grid_h}x{grid_w} grid = {len(patches_700)} patches\n({patch_size_700}x{patch_size_w} each)", 
#               fontsize=12, fontweight='bold')
# ax5.axis('off')

# # 6. Comparison: Original vs Cropped side by side
# ax6 = axes[1, 2]
# comparison = np.hstack([
#     single_channel,
#     np.ones((h, 10), dtype=np.float32),  # Separator
#     np.vstack([
#         np.zeros((crop_top, new_width), dtype=np.float32),
#         cropped_700,
#         np.zeros((crop_bottom, new_width), dtype=np.float32)
#     ])
# ])
# ax6.imshow(comparison, cmap='gray', vmin=0, vmax=1)
# ax6.set_title(f"Comparison: Original (left) vs Cropped (right)\nOriginal: {h}x{w}, Cropped: {new_height}x{new_width}", 
#               fontsize=12, fontweight='bold')
# ax6.axis('off')

# plt.tight_layout()
# plt.show()

# # Summary
# print(f"\n{'='*60}")
# print("SUMMARY: 700x576 Crop")
# print(f"{'='*60}")
# print(f"Original dimensions: {h}x{w}")
# print(f"Cropped dimensions: {new_height}x{new_width}")
# print(f"Lost: {crop_top} rows (top) + {crop_bottom} rows (bottom) = {crop_top + crop_bottom} rows")
# print(f"Lost percentage: {100 * (crop_top + crop_bottom) / h:.1f}% of height")

# if pipe_info:
#     visible_pipes = sum(1 for p in pipe_info if p['visible_x'] and p['visible_y'])
#     total_pipes = len(pipe_info)
#     print(f"\nPipe visibility:")
#     print(f"  Total pipes: {total_pipes}")
#     print(f"  Fully visible: {visible_pipes} ({100*visible_pipes/total_pipes:.1f}%)")
#     if visible_pipes < total_pipes:
#         print(f"  ⚠️  {total_pipes - visible_pipes} pipes partially or fully outside crop region")
#     else:
#         print(f"  ✓ All pipes are visible in cropped region")

# print(f"\nPatch extraction (with patch_size=140x144):")
# print(f"  Grid: {grid_h}x{grid_w} = {len(patches_700)} patches")
# print(f"  Each patch: {patch_size_700}x{patch_size_w}")

# env.close()


In [7]:
# # Create a random individual and test evaluation
# print("Creating random individual...")
# test_individual = Individual.random(
#     instruction_set, 
#     memory_cfg, 
#     program_length=20,  # Start with short programs
#     rng=rng
# )

# print(f"Program length: {len(test_individual.program.instructions)}")
# print(f"Memory shape: {test_individual.memory.obs_matrices.shape}")

# # Test evaluation (this will run episodes and show pygame window)
# print("\nRunning evaluation (watch the pygame window!)...")
# print("Note: The agent will be random, so it won't play well yet.")
# fitness = evaluator.evaluate(test_individual)
# print(f"\nFitness (average reward over {evaluator.episodes} episodes): {fitness:.2f}")

# # Close the evaluator's environment
# evaluator.close()


In [8]:
# Small Evolutionary Test
# Set up evaluator for evolution (no rendering for speed)
evolution_evaluator = FlappyBirdEvaluator(
    env_id="FlappyBird-v0",
    episodes=5,  # Fewer episodes for faster evolution
    max_steps=500,  # Shorter episodes
    output_register=15,
    render_mode="human",  # No rendering for speed
    rng=rng,
    patch_size=700,
    num_patches=8,  # Ignored for "full" strategy, but kept for compatibility
    patch_strategy="full_image",
    color_channel=1,
    normalize=True,
)

# Calculate actual number of patches for "full" strategy
# With patch_size=200 on 800x800 image: 800/200 = 4, so 4x4 = 16 patches
actual_num_patches = 1  # This matches what we saw in Cell 3

# Update memory config to match actual patches
memory_cfg_evolution = MemoryConfig(
    n_scalar=16,
    n_vector=16,
    n_matrix=16,
    n_obs_scalar=0,
    n_obs_vector=0,
    n_obs_matrix=1,  # Match actual number of patches
    vector_size=700,
    matrix_shape=(700, 700),
)

# Create instruction set with updated config
instruction_set_evolution = InstructionSet([op() for op in all_ops], memory_cfg_evolution)
operators_evolution = GeneticOperators(instruction_set_evolution, rng)

# Small population for quick test
pop_config = PopulationConfig(
    size=5,  # Small population
    program_length=(10, 100),  # Short to medium programs
    elitism=1,  # Keep best individual
    max_program_length=250,
)

# Create population
population = Population(
    pop_config,
    instruction_set_evolution,
    memory_cfg_evolution,
    operators=operators_evolution,
    rng=rng,
)
population.initialize_random(mutate_constants=True)

print("Evolution Setup Complete!")
print(f"Population size: {pop_config.size}")
print(f"Program length range: {pop_config.program_length}")
print(f"Episodes per evaluation: {evolution_evaluator.episodes}")
print(f"Max steps per episode: {evolution_evaluator.max_steps}")
print(f"Number of patches: {actual_num_patches}")


Evolution Setup Complete!
Population size: 5
Program length range: (10, 100)
Episodes per evaluation: 5
Max steps per episode: 500
Number of patches: 1


In [11]:
# Run evolutionary loop
evolution_config = EvolutionConfig(
    max_generations=100,  # Small number for quick test
    mutation_threshold=0.9,
    constant_mutation_rate=0.1,
    verbose=True,
)

engine = EvolutionEngine(
    population=population,
    operators=operators_evolution,
    evaluator=evolution_evaluator,
    config=evolution_config,
    rng=rng,
)

print("Starting evolution...")
print("=" * 60)
final_population = engine.run()
print("=" * 60)
print("\nEvolution complete!")

# Close evaluator
evolution_evaluator.close()


Starting evolution...
Best agent: fitness=0.0340, effective_code_rate=0.023 (2/87)

=== Generation 0 ===
Generation 0 | Population size 5
Min: 0.027, Mean: 0.033, Max: 0.034, Std: 0.003
Length mean 49.4, std 23.7
best agent


AttributeError: 'NoneType' object has no attribute 'program'

In [ ]:
# Display best agent history
print("\n" + "=" * 60)
print("BEST AGENT HISTORY (Fitness & Effective Code Rate)")
print("=" * 60)
print(f"{'Gen':<5} {'Fitness':<12} {'Eff. Code Rate':<18} {'Eff. Length':<15} {'Total Length':<15}")
print("-" * 60)

for info in engine.best_agent_history:
    print(f"{info.generation:<5} {info.fitness:<12.4f} {info.effective_code_rate:<18.3f} "
          f"{info.effective_length:<15} {info.total_length:<15}")

# Plot fitness and effective code rate over generations
import matplotlib.pyplot as plt

if len(engine.best_agent_history) > 0:
    generations = [info.generation for info in engine.best_agent_history]
    fitnesses = [info.fitness for info in engine.best_agent_history]
    code_rates = [info.effective_code_rate for info in engine.best_agent_history]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot fitness
    ax1.plot(generations, fitnesses, 'b-o', linewidth=2, markersize=8)
    ax1.set_xlabel('Generation', fontsize=12)
    ax1.set_ylabel('Fitness', fontsize=12)
    ax1.set_title('Best Agent Fitness Over Generations', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(-0.5, max(generations) + 0.5)
    
    # Plot effective code rate
    ax2.plot(generations, code_rates, 'r-o', linewidth=2, markersize=8)
    ax2.set_xlabel('Generation', fontsize=12)
    ax2.set_ylabel('Effective Code Rate', fontsize=12)
    ax2.set_title('Best Agent Effective Code Rate', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 1.1)
    ax2.set_xlim(-0.5, max(generations) + 0.5)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nBest overall fitness: {max(fitnesses):.4f} (Generation {generations[fitnesses.index(max(fitnesses))]})")
    print(f"Final effective code rate: {code_rates[-1]:.3f}")



BEST AGENT HISTORY (Fitness & Effective Code Rate)
Gen   Fitness      Eff. Code Rate     Eff. Length     Total Length   
------------------------------------------------------------


AttributeError: 'EvolutionEngine' object has no attribute 'best_agent_history'